<a href="https://colab.research.google.com/github/DEEK-SHITH/UNLET-ADAS/blob/main/notebooks/UNLET_ADAS_Lowlight_YOLO_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌙 UNLET-ADAS: Low-Light Detector Training (Colab, free GPU)
### B.E. Major Project | SJBIT Bengaluru | CSE 2025-26
**GitHub:** https://github.com/DEEK-SHITH/UNLET-ADAS

Fine-tunes a dedicated **5-class low-light specialist** YOLOv8 model
(Person / Bicycle / Car / Motorcycle / Bus) on ExDark (Exclusively
Dark Image Dataset — Loh & Chan, CVIU 2019), so detection is trained
on real night-time appearance instead of relying entirely on stock
COCO daylight weights run on enhanced frames.

This is a **separate model**, not a replacement for the main 8-class
ADAS detector: ExDark has no Truck / Traffic Light / Stop Sign data,
and fine-tuning directly on its 12 classes would silently drop those
3 classes rather than leave them unchanged (Ultralytics resizes the
detection head to match the training class list). The app offers this
as an alternative "Detector Model" choice alongside the stock COCO
model, which keeps full 8-class daylight coverage.

**Dataset sourcing — nothing to configure by default.** Cell 3
downloads ExDark directly from its authors' own Google Drive (linked
from https://github.com/cs-chan/Exclusively-Dark-Image-Dataset) and
converts the official annotation format straight to YOLO — no
Roboflow account, no API key, no hunting for a mirror. Non-commercial
research use only, per the dataset's license.

If Google Drive ever rate-limits or blocks the download from Colab's
IP range, Cell 2 has a `DATA_SOURCE = 'roboflow'` fallback: search
[Roboflow Universe](https://universe.roboflow.com) for "ExDark", pick
a project whose image count is close to the real dataset's ~7,363,
and paste its workspace/project/version from **Download Dataset →
YOLOv8 → Show download code**.

| Cell | What it does |
|---|---|
| 1 | Setup — install packages, mount Drive, clone GitHub |
| 2 | Configuration — data source + paths + hyperparameters |
| 3 | Download the dataset and filter it to the 5 ADAS-relevant classes |
| 4 | Train YOLOv8 low-light detector (~20–35 min on a T4 GPU) |
| 5 | Check results — metrics + sample predictions |
| 6 | Save weights + deployment instructions |

**Run cells top to bottom. Do not skip any cell.**


In [1]:
# ============================================================
# CELL 1 — Setup
# ============================================================

# Anti-disconnect — run this first
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    var btns = document.querySelectorAll("colab-toolbar-button");
    for(var i=0;i<btns.length;i++){
        if(btns[i].id=="connect") btns[i].click();
    }
}
setInterval(ClickConnect, 55000)
'''))
print('Anti-disconnect active!')

# Mount Google Drive (so trained weights survive when the Colab runtime recycles)
from google.colab import drive
drive.mount('/content/drive')

# Install packages
!pip install ultralytics gdown roboflow -q

# Clone or update GitHub repo
import os
if not os.path.exists('/content/UNLET-ADAS'):
    !git clone https://github.com/DEEK-SHITH/UNLET-ADAS.git /content/UNLET-ADAS
    print('Repo cloned!')
else:
    !cd /content/UNLET-ADAS && git pull
    print('Repo updated!')

import sys
if '/content/UNLET-ADAS' not in sys.path:
    sys.path.insert(0, '/content/UNLET-ADAS')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')
print('Setup complete!')


<IPython.core.display.Javascript object>

Anti-disconnect active!
Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.1 MB/s eta 0:00:00
Cloning into '/content/UNLET-ADAS'...
remote: Enumerating objects: 301, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 301 (delta 29), reused 47 (delta 15), pack-reused 238 (from 1)
Receiving objects: 100% (301/301), 59.22 MiB | 31.08 MiB/s, done.
Resolving deltas: 100% (160/160), done.
Updating files: 100% (33/33), done.
Repo clon

In [2]:
# ============================================================
# CELL 2 — Configuration
# All paths and settings are defined here. Edit only this cell if
# settings need to change.
# ============================================================

# 'exdark_official' (default): downloads straight from the dataset
# authors' Google Drive, no account needed. 'roboflow': use a
# hand-found Roboflow Universe mirror instead (fill in the ROBOFLOW_*
# values below) -- only needed if the official download is blocked
# from this Colab session's network.
DATA_SOURCE = 'exdark_official'

ROBOFLOW_API_KEY       = 'PASTE_YOUR_OWN_KEY_HERE'   # app.roboflow.com -> Settings -> API Keys
ROBOFLOW_WORKSPACE     = 'PASTE_WORKSPACE_HERE'       # from the "Show download code" snippet
ROBOFLOW_PROJECT       = 'PASTE_PROJECT_HERE'         # from the "Show download code" snippet
ROBOFLOW_VERSION       = 1                            # from the "Show download code" snippet

DATASET_DIR = '/content/lowlight_dataset'                             # downloaded fresh each run
SAVE_DIR    = '/content/drive/MyDrive/UNLET_Project/checkpoints'      # persists across sessions
MODEL_SIZE  = 'n'          # n = fastest/smallest, matches the lightweight theme of this project
EPOCHS      = 60
BATCH_SIZE  = 16
IMAGE_SIZE  = 640
PATIENCE    = 15           # early stop if val mAP doesn't improve for this many epochs

os.makedirs(SAVE_DIR, exist_ok=True)

if DATA_SOURCE == 'roboflow':
    _missing = [n for n, v in [
        ('ROBOFLOW_API_KEY', ROBOFLOW_API_KEY),
        ('ROBOFLOW_WORKSPACE', ROBOFLOW_WORKSPACE),
        ('ROBOFLOW_PROJECT', ROBOFLOW_PROJECT),
    ] if v.startswith('PASTE_')]
    if _missing:
        print(f'MISSING: set {", ".join(_missing)} above before continuing.')
    else:
        print('Configuration OK (Roboflow source). Ready for Cell 3.')
else:
    print('Configuration OK (official ExDark source, no account needed). Ready for Cell 3.')
print(f'Save dir : {SAVE_DIR}')


Configuration OK (official ExDark source, no account needed). Ready for Cell 3.
Save dir : /content/drive/MyDrive/UNLET_Project/checkpoints


In [3]:
# ============================================================
# CELL 3 — Download the ExDark dataset, then filter it down to the 5
# ADAS-relevant classes (Person/Bicycle/Car/Motorcycle/Bus), remapped
# to ids 0..4. Watch the printed box-count report -- if any of the 5
# classes shows 0 boxes, something's wrong with the download; re-run
# this cell, or switch DATA_SOURCE in Cell 2 and try again.
# ============================================================

if DATA_SOURCE == 'roboflow':
    from src.train_lowlight import download_dataset, filter_and_remap_dataset

    print('Downloading ExDark dataset from Roboflow...')
    raw_location = download_dataset(
        ROBOFLOW_API_KEY, DATASET_DIR,
        ROBOFLOW_WORKSPACE, ROBOFLOW_PROJECT, ROBOFLOW_VERSION)
    print(f'Raw dataset at: {raw_location}')

    print('\nFiltering to ADAS-relevant classes and remapping IDs...')
    filtered_dir = os.path.join(DATASET_DIR, 'filtered')
    data_yaml = filter_and_remap_dataset(raw_location, filtered_dir)
else:
    from src.train_lowlight import download_exdark_official, convert_exdark_official

    print('Downloading the official ExDark dataset (images + groundtruth, '
          "direct from the authors' Google Drive)... this is ~1.5GB and "
          'can take a few minutes.')
    images_root, groundtruth_root = download_exdark_official(DATASET_DIR)
    print(f'Images     : {images_root}')
    print(f'Groundtruth: {groundtruth_root}')

    print('\nConverting to YOLO format, filtering to ADAS-relevant classes '
          'and remapping IDs...')
    filtered_dir = os.path.join(DATASET_DIR, 'filtered')
    data_yaml = convert_exdark_official(images_root, groundtruth_root, filtered_dir)

print(f'\nFiltered dataset ready at: {filtered_dir}')
print(f'data.yaml                : {data_yaml}')
assert os.path.exists(data_yaml), 'data.yaml not found — check the output above for errors.'


Downloading...
From (original): https://drive.google.com/uc?id=1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx
From (redirected): https://drive.google.com/uc?id=1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx&confirm=t&uuid=4bd97262-4726-47b6-b60e-dcb672f7ba06
To: /content/lowlight_dataset/images.zip
100%|██████████| 1.49G/1.49G [00:29<00:00, 51.1MB/s]


Extracting images...


Downloading...
From (original): https://drive.google.com/uc?id=1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i
From (redirected): https://drive.google.com/uc?id=1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i&confirm=t&uuid=628cdc38-d436-4164-b9a4-475b4cdf24fc
To: /content/lowlight_dataset/groundtruth.zip
100%|██████████| 5.08M/5.08M [00:00<00:00, 52.8MB/s]


Extracting groundtruth...
Images     : /content/lowlight_dataset/images/ExDark
Groundtruth: /content/lowlight_dataset/groundtruth/ExDark_Annno

Converting to YOLO format, filtering to ADAS-relevant classes and remapping IDs...

Images processed: 7363, kept: 7363, skipped (missing/unreadable file): 0
Boxes per source class (all 12, before filtering): {'Bicycle': 1120, 'Car': 2927, 'Bus': 706, 'Chair': 2377, 'People': 7460, 'Motorbike': 1072, 'Dog': 1018, 'Table': 1483, 'Bottle': 1593, 'Boat': 1389, 'Cat': 909, 'Cup': 1656}
Boxes per ADAS class (after filtering+remapping): {'Person': 7460, 'Bicycle': 1120, 'Car': 2927, 'Motorcycle': 1072, 'Bus': 706}

Filtered dataset ready at: /content/lowlight_dataset/filtered
data.yaml                : /content/lowlight_dataset/filtered/data.yaml


In [ ]:
# ============================================================
# CELL 4 — Train YOLOv8 low-light detector (~20-35 min on a T4 GPU)
# ============================================================

from ultralytics import YOLO

model = YOLO(f'yolov8{MODEL_SIZE}.pt')
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    project=SAVE_DIR,
    name='lowlight_run',
    exist_ok=True,
)

print('\nTraining complete!')


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/lowlight_dataset/filtered/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, f

In [ ]:
# ============================================================
# CELL 5 — Check results: metrics + sample predictions
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = os.path.join(SAVE_DIR, 'lowlight_run')

# Training curves (loss / mAP / precision / recall over epochs)
results_png = os.path.join(run_dir, 'results.png')
if os.path.exists(results_png):
    img = mpimg.imread(results_png)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training curves')
    plt.show()

# Sample validation predictions (model's own boxes drawn on val images)
val_pred = os.path.join(run_dir, 'val_batch0_pred.jpg')
if os.path.exists(val_pred):
    img = mpimg.imread(val_pred)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Sample validation predictions')
    plt.show()

best_weights = os.path.join(run_dir, 'weights', 'best.pt')
print(f'\nBest weights: {best_weights}')
print(f'Exists      : {os.path.exists(best_weights)}')


In [ ]:
# ============================================================
# CELL 6 — Save weights + deployment instructions
# ============================================================

import shutil

best_weights = os.path.join(SAVE_DIR, 'lowlight_run', 'weights', 'best.pt')
final_path   = os.path.join(SAVE_DIR, 'lowlight_best.pt')

if os.path.exists(best_weights):
    shutil.copy(best_weights, final_path)
    print(f'Saved to Google Drive: {final_path}')
    print()
    print('To enable the Low-Light Detector option in the Streamlit app:')
    print('  1. Download this file from your Google Drive.')
    print('  2. Rename it to: yolov8_lowlight.pt')
    print('  3. Place it at: app/yolov8_lowlight.pt   (in your local UNLET-ADAS clone)')
    print('  4. Restart the Streamlit app -- the "Detector Model" dropdown will gain a')
    print('     "yolov8_lowlight.pt (fine-tuned, ExDark)" option. Note it only detects')
    print('     Person/Bicycle/Car/Motorcycle/Bus -- no Traffic Light/Stop Sign/Truck.')
else:
    print(f'Expected weights at {best_weights} but they were not found -- '
          'check Cell 4\'s training log for errors.')
